# Observation quality and provenance

This notebook checks the composition calendar and the integrity of archived observation arrays. It reads the local catalogue and acquisition receipts; it does not download satellite assets. The array checks require the offline-input archive.

In [ ]:
from pathlib import Path
import os,sys,json
import pandas as pd
from IPython.display import display,SVG
from terraclass_ppi.paths import CODE_ROOT as ROOT,output_root
from terraclass_ppi.data_io import prepare_reference,table_directory,iter_candidate_groups
WORK=Path(os.environ['RBC_OUTPUT']).resolve() if os.environ.get('RBC_OUTPUT') else prepare_reference()
os.environ['RBC_OUTPUT']=str(WORK)
FIGURES=output_root()/'figures'
from terraclass_ppi.spatial import CFG


## Apply the quality fields

The Scene Classification Layer (SCL) identifies vegetation (4), non-vegetated surface (5), water (6) and unclassified observations (7), the four categories retained before spectral classification. CLEAROB counts clear observations; PROVENANCE identifies the acquisition used in the composite. A pixel is eligible on a date when CLEAROB > 0, SCL is 4–7 and all ten band values are valid. Source integer values are converted to reflectance with factor 10⁻⁴.

The structured NumPy arrays retain these source fields and a validity flag. Receipt hashes allow integrity checks without holding the full time series in memory. A missing or damaged input file is an execution error. Coverage records separately identify dates with no valid pixel, too few pixels to process, or no accepted spectral candidate.

In [ ]:
from terraclass_ppi.spatial import sha
catalog=pd.read_parquet(WORK/'catalog.parquet')
counts=catalog.groupby('tile_id').date.nunique()
assert len(counts)==2 and counts.eq(184).all()
display(catalog.groupby('tile_id').date.agg(['min','max','size']))
receipts=sorted((WORK/'observations').glob('*.json'))
if receipts:
    expected_dates=set(pd.to_datetime(catalog.date).dt.strftime('%Y-%m-%d'))
    assert {p.stem for p in receipts}==expected_dates, 'Observation inventory is incomplete'
    for p in receipts:
        info=json.loads(p.read_text());assert sha(p.with_suffix('.npy'))==info['sha256']
    print(len(receipts),'observation arrays verified')
else: print('Quick reference route: observation arrays belong to the separate offline-input archive.')
display(pd.read_csv(WORK/'results/coverage-13.csv').groupby(['classe','state']).size())


## Interpret availability

The catalogue contains 184 dates for each tile. There are 33 non-processable class–date combinations across the three classes: 31 have no valid pixel and two have fewer than 101. If the observation archive is absent, this notebook reports that condition and checks only the catalogue and coverage tables; it has not verified array integrity in that case.

## References

Brazil Data Cube collection and source assets: `work/catalog.parquet`; quality rules: `docs/METHODOLOGY.md`.